# GRU — the Gated Recurrent Unit

> Tutorial pair for [`gru.py`](gru.py). Read [`rnn.ipynb`](rnn.ipynb) and
> [`lstm.ipynb`](lstm.ipynb) first.

## 1. Intuition
The LSTM beats vanishing gradients with a separate cell state and **three** gates.
The **GRU** (Cho et al., 2014) asks: can we get the same long-memory behaviour
more cheaply? It keeps a **single** hidden state and just **two** gates — a
*reset* gate and an *update* gate. The update gate linearly blends the old state
with a fresh candidate, so when it decides to "carry", the state (and its
gradient) passes through almost untouched.

## 2. Concept (the slide)
At each step (with $z=[x_t,h_{t-1}]$):
- **reset gate** $r_t=\sigma(\cdot)$ — how much of the past to *mix into* the
  candidate (when $r_t\approx0$ the candidate ignores history),
- **update gate** $u_t=\sigma(\cdot)$ — how much of the past state to *keep*,
- **candidate** $n_t=\tanh(\cdot)$ — the proposed new content (sees $r_t\odot h_{t-1}$),
- **state** $h_t=(1-u_t)\odot n_t + u_t\odot h_{t-1}$ — a **convex combination**.

No separate cell state, no output gate. Fewer parameters than the LSTM (2 gates
vs 3), often comparable accuracy.

## 3. Math derivation — why the update gate is a gradient highway

**Equations.** With $z_t=[x_t,h_{t-1}]$ and candidate input $\tilde z_t=[x_t,\,r_t\odot h_{t-1}]$:
$$
r_t=\sigma(W_r z_t+b_r),\qquad
u_t=\sigma(W_u z_t+b_u),
$$
$$
n_t=\tanh(W_n \tilde z_t + b_n),\qquad
h_t=(1-u_t)\odot n_t + u_t\odot h_{t-1}.
$$

**The carry path.** Differentiate the state update w.r.t. the previous state. Two
terms appear — a *direct* copy term and an *indirect* term through the candidate:
$$
\frac{\partial h_t}{\partial h_{t-1}}
=\underbrace{\operatorname{diag}(u_t)}_{\text{direct carry}}
+\;\underbrace{\operatorname{diag}(1-u_t)\,\frac{\partial n_t}{\partial h_{t-1}}}_{\text{through candidate}} .
$$
When the update gate stays open ($u_t\approx1$) the second term is suppressed by
$1-u_t\approx0$ and the Jacobian collapses to
$$
\frac{\partial h_t}{\partial h_{t-1}}\approx\operatorname{diag}(u_t)\approx I,
$$
so over $k$ steps the product is $\prod u\approx 1$ — the gradient flows back
**unattenuated**. This is the same idea as the LSTM's *constant error carousel*
($\partial c_t/\partial c_{t-1}=\operatorname{diag}(f_t)$), and contrasts sharply
with the vanilla RNN's $\operatorname{diag}(1-h^2)W_{hh}^\top$, whose repeated
multiplication shrinks (vanishing) or grows (exploding) geometrically.

**Comparison at a glance:**

| cell | carry Jacobian | states / gates |
|---|---|---|
| RNN | $\operatorname{diag}(1-h_t^2)\,W_{hh}^\top$ (decays) | 1 / 0 |
| LSTM | $\operatorname{diag}(f_t)$ on $c_t$ | 2 / 3 |
| GRU | $\operatorname{diag}(u_t)$ on $h_t$ | 1 / 2 |

**Backprop (BPTT) sketch.** Let $\delta h_t=\partial\mathcal L/\partial h_t$. From
$h_t=(1-u_t)n_t+u_t h_{t-1}$:
$$
\delta n_t=\delta h_t\odot(1-u_t),\quad
\delta u_t=\delta h_t\odot(h_{t-1}-n_t),\quad
\delta h_{t-1}\mathrel{+}=\delta h_t\odot u_t .
$$
Push $\delta n_t$ through $\tanh$ ($\times(1-n_t^2)$) into $W_n$ and into
$r_t\odot h_{t-1}$ (adding $\delta r_t = (\cdot)\odot h_{t-1}$ and another
contribution $(\cdot)\odot r_t$ to $\delta h_{t-1}$); push $\delta u_t,\delta r_t$
through their sigmoids ($\times g(1-g)$) into $W_u,W_r$ and back into
$\delta h_{t-1}$. The module codes every one of these terms by hand.

## 4. NumPy implementation — GRU cell + BPTT by hand

In [ ]:
# ===== actual implementation from gru.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))

def softmax(z):
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

class GRUNumPy:
    r"""
    Gates (z_t = [x_t, h_{t-1}] concatenated for the gate pre-activations):
        r_t = σ(W_r [x_t, h_{t-1}] + b_r)      reset gate  — how much past to forget
        u_t = σ(W_u [x_t, h_{t-1}] + b_u)      update gate — how much past to keep
        n_t = tanh(W_n [x_t, (r_t ⊙ h_{t-1})] + b_n)   candidate state
    State (a CONVEX COMBINATION — this is the gradient highway):
        h_t = (1 - u_t) ⊙ n_t + u_t ⊙ h_{t-1}

    The carry path ∂h_t/∂h_{t-1} contains an explicit additive term diag(u_t):
    when the update gate stays near 1 the state is copied through and the
    gradient passes back essentially unattenuated — like the LSTM's constant
    error carousel, but with one state vector and two gates instead of three.

    We read out the LAST hidden state (many-to-one):  logits = h_T W_hy + b_y.
    """

    def __init__(self, in_dim, hidden, out_dim, lr=0.1, clip=5.0, seed=SEED):
        rng = np.random.default_rng(seed)
        Z = in_dim + hidden
        s = 1.0 / np.sqrt(Z)
        # Each gate sees [x_t, h_{t-1}] (or [x_t, r⊙h] for the candidate).
        self.W = {k: rng.normal(0, s, (Z, hidden)) for k in ("r", "u", "n")}
        self.b = {k: np.zeros(hidden) for k in ("r", "u", "n")}
        self.Why = rng.normal(0, 1 / np.sqrt(hidden), (hidden, out_dim))
        self.by = np.zeros(out_dim)
        self.H, self.in_dim, self.lr, self.clip = hidden, in_dim, lr, clip

    def forward(self, X):
        """X: (T, in_dim). Returns softmax over the last-step logits."""
        T = len(X)
        H = self.H
        self.cache = []
        h = np.zeros(H)
        for t in range(T):
            xz = np.concatenate([X[t], h])              # [x_t, h_{t-1}]
            r = sigmoid(xz @ self.W["r"] + self.b["r"])  # reset gate
            u = sigmoid(xz @ self.W["u"] + self.b["u"])  # update gate
            rn = np.concatenate([X[t], r * h])           # candidate sees reset state
            n = np.tanh(rn @ self.W["n"] + self.b["n"])  # candidate
            h_new = (1 - u) * n + u * h                  # convex combination
            self.cache.append((X[t], h, r, u, n, rn, h_new))
            h = h_new
        self.logits = h @ self.Why + self.by
        return softmax(self.logits)

    def backward(self, y):
        T = len(self.cache)
        H = self.H
        p = softmax(self.logits)
        p[y] -= 1.0                                      # dL/dlogits
        dW = {k: np.zeros_like(self.W[k]) for k in ("r", "u", "n")}
        db = {k: np.zeros_like(self.b[k]) for k in ("r", "u", "n")}
        h_last = self.cache[-1][6]
        dWhy = np.outer(h_last, p)
        dby = p.copy()
        dh = p @ self.Why.T                              # grad into last hidden state
        self.bptt_norms = []                             # ||dh|| over time
        for t in reversed(range(T)):
            x_t, h_prev, r, u, n, rn, h_new = self.cache[t]
            self.bptt_norms.append(np.linalg.norm(dh))
            # h_t = (1 - u) n + u h_prev
            dn = dh * (1 - u)
            du = dh * (h_prev - n)
            dh_prev = dh * u                             # <-- the gradient highway term
            # candidate n = tanh(W_n [x, r⊙h_prev])
            da_n = dn * (1 - n ** 2)                     # pre-activation grad
            dW["n"] += np.outer(rn, da_n)
            db["n"] += da_n
            drn = da_n @ self.W["n"].T                   # grad into [x, r⊙h_prev]
            d_rh = drn[self.in_dim:]                     # grad into (r ⊙ h_prev)
            dr = d_rh * h_prev
            dh_prev += d_rh * r                          # path through candidate
            # update gate u = σ(W_u [x, h_prev])
            da_u = du * u * (1 - u)
            xz = np.concatenate([x_t, h_prev])
            dW["u"] += np.outer(xz, da_u)
            db["u"] += da_u
            dh_prev += (da_u @ self.W["u"].T)[self.in_dim:]
            # reset gate r = σ(W_r [x, h_prev])
            da_r = dr * r * (1 - r)
            dW["r"] += np.outer(xz, da_r)
            db["r"] += da_r
            dh_prev += (da_r @ self.W["r"].T)[self.in_dim:]
            dh = dh_prev                                 # propagate one step back
        self.bptt_norms.reverse()
        grads = (dW, db, dWhy, dby)
        # global-norm gradient clipping
        flat = [g for d in (dW, db) for g in d.values()] + [dWhy, dby]
        total = np.sqrt(sum((g ** 2).sum() for g in flat))
        if self.clip is not None and total > self.clip:
            scale = self.clip / total
            for d in (dW, db):
                for k in d:
                    d[k] *= scale
            dWhy *= scale
            dby *= scale
        return grads

    def step(self, grads):
        dW, db, dWhy, dby = grads
        for k in ("r", "u", "n"):
            self.W[k] -= self.lr * dW[k]
            self.b[k] -= self.lr * db[k]
        self.Why -= self.lr * dWhy
        self.by -= self.lr * dby

    def fit(self, seqs, labels, epochs=40):
        self.history = []
        for _ in range(epochs):
            loss = 0.0
            for X, y in zip(seqs, labels):
                p = self.forward(X)
                loss += -np.log(p[y] + 1e-12)
                self.step(self.backward(y))
            self.history.append(loss / len(seqs))
        return self

    def predict(self, seqs):
        return np.array([self.forward(X).argmax() for X in seqs])

## 5. PyTorch implementation — idiomatic `nn.GRU` + clipping

In [ ]:
# ===== actual implementation from gru.py =====
import torch

import torch.nn as nn

def get_device():
    """CUDA > MPS > CPU. See docs/gpu-setup.md."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class GRUTorch(nn.Module):
    """Idiomatic GRU classifier (many-to-one) using PyTorch's fused `nn.GRU`."""

    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.rnn = nn.GRU(in_dim, hidden, batch_first=True)
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, x):                                # x: (B, T, in_dim)
        out, _ = self.rnn(x)
        return self.head(out[:, -1])                     # last time-step

    def fit(self, seqs, labels, epochs=60, lr=0.01, clip=5.0):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(np.array(seqs), dtype=torch.float32, device=dev)
        y = torch.as_tensor(labels, dtype=torch.long, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()
        self.history = []
        for _ in range(epochs):
            opt.zero_grad()
            loss = loss_fn(self(X), y)
            loss.backward()
            nn.utils.clip_grad_norm_(self.parameters(), clip)   # gradient clipping
            opt.step()
            self.history.append(loss.item())
        return self

    @torch.no_grad()
    def predict(self, seqs):
        dev = next(self.parameters()).device
        X = torch.as_tensor(np.array(seqs), dtype=torch.float32, device=dev)
        return self(X).argmax(1).cpu().numpy()

def make_memory_task(n=300, T=25, seed=SEED):
    """Label = identity of the FIRST token; the rest is noise. Needs long memory.

    Copied locally so this module is self-contained (no sibling imports).
    """
    rng = np.random.default_rng(seed)
    seqs, labels = [], []
    for _ in range(n):
        first = rng.integers(0, 2)
        x = rng.normal(0, 0.3, size=(T, 2))
        x[0, first] += 2.0                               # signal only at t=0
        seqs.append(x)
        labels.append(first)
    return np.array(seqs), np.array(labels)

def seed_everything(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    # On this CPU box, small RNN ops suffer heavy thread-oversubscription overhead
    # (multi-thread BLAS on tiny matmuls is far slower); one thread keeps the demo
    # well under the time budget. Harmless for these toy-sized tensors.
    torch.set_num_threads(1)

def demo():
    # Kept deliberately small (short sequences, few epochs) so the whole demo
    # runs in well under 30 s on CPU — PyTorch's CPU GRU has no cuDNN fast path.
    seed_everything(SEED)
    seqs, labels = make_memory_task(n=160, T=12)
    tr, te = slice(0, 120), slice(120, 160)

    net = GRUNumPy(2, 16, 2, lr=0.2).fit(seqs[tr], labels[tr], epochs=40)
    acc_np = np.mean(net.predict(seqs[te]) == labels[te])
    print(f"NumPy GRU   test acc = {acc_np:.3f}")

    m = GRUTorch(2, 16, 2).fit(seqs[tr], labels[tr], epochs=40)
    acc_pt = np.mean(m.predict(seqs[te]) == labels[te])
    print(f"Torch GRU   test acc = {acc_pt:.3f}")

    print("\nThe update gate u_t lets the GRU COPY the hidden state across the steps")
    print("(h_t = (1-u)·n + u·h_{t-1}); the carry term diag(u_t) is the gradient")
    print("highway — like the LSTM cell state, but with two gates and one state.")

## 6. Train / run — the long-memory toy task

In [ ]:
demo()

## 7. Visualization — the update gate holds the memory open

In [ ]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gru as M

M.seed_everything()
seqs, labels = M.make_memory_task(n=160, T=12)
net = M.GRUNumPy(2, 16, 2, lr=0.2).fit(seqs[:120], labels[:120], epochs=40)

# Run one sequence through and read the gates out of the cache.
net.forward(seqs[0])
U = np.array([c[3].mean() for c in net.cache])   # mean update-gate activation
R = np.array([c[2].mean() for c in net.cache])   # mean reset-gate activation

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(net.history)
ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss")
ax1.set_title("GRU training loss")
ax1.grid(True, alpha=.3)

ax2.plot(U, "o-", label="update gate ⟨u_t⟩  (carry)")
ax2.plot(R, "s-", label="reset gate ⟨r_t⟩")
ax2.axvline(0, ls="--", c="k", alpha=.4, label="signal at t=0")
ax2.set_xlabel("time step"); ax2.set_ylabel("mean gate activation")
ax2.set_ylim(0, 1)
ax2.set_title("u_t≈1 ⇒ state copied forward (gradient highway)")
ax2.legend()
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Convex-combination update** $h_t=(1-u_t)n_t+u_t h_{t-1}$ gives the carry
  Jacobian $\operatorname{diag}(u_t)$ — a gradient highway, like the LSTM cell but
  with one state and two gates.
- **GRU vs LSTM:** fewer parameters and often as accurate; the LSTM's extra
  output gate can help on some tasks. Try both.
- **Pitfall — the reset gate placement:** the candidate sees $r_t\odot h_{t-1}$
  *before* the matmul $W_n$, not after. Getting this wrong silently breaks the
  reset gate (and the gradient `d(r⊙h)/dr = h_{t-1}` term in BPTT).
- **Pitfall — gate saturation:** if $u_t$ saturates near 0 early, the model
  behaves like a plain RNN and gradients vanish; clipping + good init help.
- **Next:** drop recurrence entirely and attend directly to every step →
  [Transformers](../../transformers/architectures/transformer.ipynb).